# Testing Notebook

# Borzoi Embeddings

In [6]:
from borzoi_pytorch import Borzoi

borzoi = Borzoi.from_pretrained("johahi/borzoi-replicate-0")  # 'johahi/borzoi-replicate-[0-3][-mouse]'

In [22]:
import random

import torch
import torch.nn.functional as F
from borzoi_pytorch import Borzoi
from borzoi_pytorch.config_borzoi import BorzoiConfig


# 1) Generate a random DNA string of length 524,288
def random_dna(length: int) -> str:
    """Generate a random DNA string of given length."""
    return "".join(random.choices(["A", "C", "G", "T"], k=length))


dna_str = random_dna(256_128)
print("Generated random DNA length:", len(dna_str))  # should be 524288

# 2) Convert that DNA string into a one-hot tensor of shape (1, 4, 524288)
char2idx = {"A": 0, "C": 1, "G": 2, "T": 3}
indices = torch.empty((1, 256_128), dtype=torch.long)
for i, base in enumerate(dna_str):
    indices[0, i] = char2idx[base]

# one_hot_temp: (1, 524288, 4) → permute → (1, 4, 524288)
one_hot_temp = F.one_hot(indices, num_classes=4)
one_hot = one_hot_temp.permute(0, 2, 1).float()
print("one_hot.shape:", one_hot.shape)  # (1, 4, 524288)

# 3) Load Borzoi and run get_embs_after_crop

cfg = BorzoiConfig.from_pretrained("johahi/borzoi-replicate-0")
cfg.bins_to_return = 8000
borzoi = Borzoi.from_pretrained("johahi/borzoi-replicate-0", config=cfg).eval()
with torch.no_grad():
    embs = borzoi.get_embs_after_crop(one_hot)

print("Embeddings shape:", embs.shape)  # e.g. (1, config.dim, 16352)

Generated random DNA length: 256128
one_hot.shape: torch.Size([1, 4, 256128])
Embeddings shape: torch.Size([1, 1536, 8000])


# Evo2 Embeddings - requires a bit more work to get all of the packages aligned

In [ ]:
# import torch
# from evo2 import Evo2

# evo2_model = Evo2('evo2_7b')

# sequence = 'ACGT'
# input_ids = torch.tensor(
#     evo2_model.tokenizer.tokenize(sequence),
#     dtype=torch.int,
# ).unsqueeze(0).to('cuda:0')

# layer_name = 'blocks.28.mlp.l3'

# outputs, embeddings = evo2_model(input_ids, return_embeddings=True, layer_names=[layer_name])

# print('Embeddings shape: ', embeddings[layer_name].shape)

ModuleNotFoundError: No module named 'evo2'

# ChemBERTa

In [9]:
import torch
from transformers import AutoTokenizer, AutoModel

# 2a) Specify the model name on Hugging Face Hub
MODEL_NAME = "seyonec/ChemBERTa-zinc-base-v1"

# 2b) Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 2c) Load the (masked‐LM / embedding) model
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()  # we’re doing inference only

RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(767, 768, padding_idx=1)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-5): 6 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (dropou

In [10]:
# Example SMILES strings
smiles_list = ["CCO", "c1ccccc1"]

# 3a) Tokenize with padding/truncation to a uniform length
#     ChemBERTa’s tokenizer will split into character‐level tokens,
#     add special [CLS]/[SEP], etc.
encodings = tokenizer(smiles_list, padding=True, truncation=True, return_tensors="pt")
# encodings is a dict: { "input_ids": Tensor(batch_size×seq_len),
#                        "attention_mask": Tensor(batch_size×seq_len) }

In [11]:
# 4a) Move to GPU if available (optional)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
encodings = {k: v.to(device) for k, v in encodings.items()}

# 4b) Forward pass (no gradient needed)
with torch.no_grad():
    outputs = model(**encodings)
    # AutoModel returns a BaseModelOutput with:
    #   last_hidden_state shape = (batch_size, seq_len, hidden_dim)
    #   (there may be other fields—e.g. pooler_output—depending on the config)
    last_hidden = outputs.last_hidden_state

print("last_hidden_state.shape =", last_hidden.shape)

last_hidden_state.shape = torch.Size([2, 6, 768])


In [12]:
# Suppose cls_embedding = hidden at index 0 for each sequence
cls_embeddings = last_hidden[:, 0, :]  # shape: (batch_size, 768)

# Or mean‐pool across the seq_len dimension (ignoring padding via attention_mask)
attention_mask = encodings["attention_mask"].unsqueeze(-1)  # (batch_size, seq_len, 1)
masked_hidden = last_hidden * attention_mask  # zero‐out padded positions
sum_hidden = masked_hidden.sum(dim=1)  # sum over seq_len → (batch_size, 768)
lengths = attention_mask.sum(dim=1)  # how many real tokens (batch_size, 1)
mean_embeddings = sum_hidden / lengths  # (batch_size, 768)

print("CLS‐based embedding:", cls_embeddings.shape)
print("Mean‐pooled embedding:", mean_embeddings.shape)

CLS‐based embedding: torch.Size([2, 768])
Mean‐pooled embedding: torch.Size([2, 768])


In [13]:
import torch
from transformers import AutoModel, AutoTokenizer

model = AutoModel.from_pretrained("ibm/MoLFormer-XL-both-10pct", deterministic_eval=True, trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained("ibm/MoLFormer-XL-both-10pct", trust_remote_code=True)

smiles = ["Cn1c(=O)c2c(ncn2C)n(C)c1=O", "CC(=O)Oc1ccccc1C(=O)O"]
inputs = tokenizer(smiles, padding=True, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)
outputs.pooler_output

config.json:   0%|          | 0.00/1.01k [00:00<?, ?B/s]

configuration_molformer.py:   0%|          | 0.00/7.60k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ibm/MoLFormer-XL-both-10pct:
- configuration_molformer.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_molformer.py:   0%|          | 0.00/39.3k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ibm/MoLFormer-XL-both-10pct:
- modeling_molformer.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

A new version of the following files was downloaded from https://huggingface.co/ibm/MoLFormer-XL-both-10pct:
- tokenization_molformer.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/ibm/MoLFormer-XL-both-10pct:
- tokenization_molformer_fast.py
- tokenization_molformer.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


vocab.json:   0%|          | 0.00/41.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/54.0k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

tensor([[-0.4407,  0.3902,  0.7989,  ..., -0.6081, -0.0200,  0.0103],
        [ 0.5943,  0.4527,  0.3437,  ...,  0.1520, -0.3464,  0.5590]])

# ESM C

In [1]:
print("blablabla")

blablabla


In [5]:
# from huggingface_hub import login
# from esm.models.esm3 import ESM3
# from esm.sdk.api import ESM3InferenceClient, ESMProtein, GenerationConfig

# # Will instruct you how to get an API key from huggingface hub, make one with "Read" permission.
# login()

# # This will download the model weights and instantiate the model on your machine.
# model: ESM3InferenceClient = ESM3.from_pretrained("esm3-open").to("cpu") # or "cpu"

# # Generate a completion for a partial Carbonic Anhydrase (2vvb)
# prompt = "___________________________________________________DQATSLRILNNGHAFNVEFDDSQDKAVLKGGPLDGTYRLIQFHFHWGSLDGQGSEHTVDKKKYAAELHLVHWNTKYGDFGKAVQQPDGLAVLGIFLKVGSAKPGLQKVVDVLDSIKTKGKSADFTNFDPRGLLPESLDYWTYPGSLTTPP___________________________________________________________"
# protein = ESMProtein(sequence=prompt)
# # Generate the sequence, then the structure. This will iteratively unmask the sequence track.
# protein = model.generate(protein, GenerationConfig(track="sequence", num_steps=8, temperature=0.7))
# # We can show the predicted structure for the generated sequence.
# protein = model.generate(protein, GenerationConfig(track="structure", num_steps=8))
# protein.to_pdb("./generation.pdb")
# # Then we can do a round trip design by inverse folding the sequence and recomputing the structure
# protein.sequence = None
# protein = model.generate(protein, GenerationConfig(track="sequence", num_steps=8))
# protein.coordinates = None
# protein = model.generate(protein, GenerationConfig(track="structure", num_steps=8))
# protein.to_pdb("./round_tripped.pdb")

# Testing Out Embeddings

In [ ]:
import torch
import numpy as np

# 2) Import your BioEmbedder (which under the hood will use EnformerWrapper)
from embpy.embedder import BioEmbedder


# 1) Generate a random DNA string (you can choose any length; Enformer will pad or truncate to 196 608 bp)
def random_dna(length: int) -> str:  # noqa: D103
    return "".join(random.choices(["A", "C", "G", "T"], k=length))


# For example, pick length = 200 000 (slightly above 196 608 so Enformer will center‐crop):
dna_seq = random_dna(200_000)
print("Generated DNA length:", len(dna_seq))  # → 200000


# 3) Instantiate BioEmbedder and load the Enformer model onto CPU (or "cuda" if you have a GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
embedder = BioEmbedder(device=device)

# 4) Now call embed_gene with a dummy “gene identifier”—but since we already have the raw DNA,
#    we’ll bypass gene‐resolution and call the EnformerWrapper directly.
#    (Alternatively, if your GeneResolver can fetch a DNA string for an actual gene, you can use embed_gene.)

#   Here we manually get the Enformer wrapper from the embedder’s registry, then call embed(…).
model_name = "enformer_human_rough"
wrapper = embedder._get_model(model_name)  # this will load under the hood if not already loaded

# 5) Embed the random DNA sequence using Enformer
#    (EnformerWrapper will pad/truncate internally to exactly 196608, one-hot encode, run the model, pool)
embedding = wrapper.embed(input=dna_seq, pooling_strategy="mean")

print("Enformer embedding shape:", embedding.shape)
# Should be (3072,) because EnformerWrapper.TRUNK_OUTPUT_DIM = 3072

# 6) (Optional) Convert to float32 NumPy if needed:
embedding = embedding.astype(np.float32)

Generated DNA length: 200000


Enformer embedding shape: (3072,)


In [ ]:
import torch
import numpy as np
from embpy.embedder import BioEmbedder


# 1) Generate a random DNA string (you can choose any length; Enformer will pad or truncate to 196 608 bp)
def random_dna(length: int) -> str:  # noqa: D103
    return "".join(random.choices(["A", "C", "G", "T"], k=length))


# For example, pick length = 200 000 (slightly above 196 608 so Enformer will center‐crop):
dna_seq = random_dna(196_608)
print("Generated DNA length:", len(dna_seq))  # → 200000

# 2) Import your BioEmbedder (which under the hood will use EnformerWrapper)

# 3) Instantiate BioEmbedder and load the Enformer model onto CPU (or "cuda" if you have a GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
embedder = BioEmbedder(device=device)

# 4) Now call embed_gene with a dummy “gene identifier”—but since we already have the raw DNA,
#    we’ll bypass gene‐resolution and call the EnformerWrapper directly.
#    (Alternatively, if your GeneResolver can fetch a DNA string for an actual gene, you can use embed_gene.)

#   Here we manually get the Enformer wrapper from the embedder’s registry, then call embed(…).
model_name = "borzoi_v0"
wrapper = embedder._get_model(model_name)  # this will load under the hood if not already loaded

# 5) Embed the random DNA sequence using Enformer
#    (EnformerWrapper will pad/truncate internally to exactly 196608, one-hot encode, run the model, pool)
embedding = wrapper.embed(input=dna_seq, pooling_strategy="mean")

print("Borzoi embedding shape:", embedding.shape)
# Should be (3072,) because EnformerWrapper.TRUNK_OUTPUT_DIM = 3072

# 6) (Optional) Convert to float32 NumPy if needed:
embedding = embedding.astype(np.float32)

Generated DNA length: 196608
Running Borzoi model …
torch.Size([1, 1536, 16352])
Borzoi embedding shape: (1536,)


# ESM 2 embeddings 

In [2]:
# In [1]:
from embpy.embedder import BioEmbedder
import torch

# 1) Initialize your embedder on CPU (or "cuda"):
embedder = BioEmbedder(device=torch.device("cpu"))

# 2) See which models you have available:
print("Available models:", embedder.list_available_models())

# 3) Pick an ESM2 model key from that list:
model_key = "esm2_650M"

# 4) Generate a random protein (amino‐acid) sequence:
amino_acids = "ACDEFGHIKLMNPQRSTVWY"
seq_length = 200
random_protein = "".join(random.choices(amino_acids, k=seq_length))
print(f"Random protein ({seq_length} aa):", random_protein[:50] + "…")

# 5) Load the wrapper (this will download & cache the HF weights under-the-hood):
wrapper = embedder._get_model(model_key)

# 6) Embed your sequence (default is “mean” pooling; you can also use “max” or “cls”):
embedding = wrapper.embed(random_protein, pooling_strategy="mean")

# 7) Inspect the result:
print("Embedding shape:", embedding.shape)  # e.g. (768,) or however large your ESM-2 hidden dim is

Available models: ['bert_base_uncased', 'borzoi_v0', 'borzoi_v1', 'chemberta_zinc_v1', 'enformer_human_rough', 'esm2_150M', 'esm2_35M', 'esm2_3B', 'esm2_650M', 'esm2_8M', 'minilm_l6_v2']
Random protein (200 aa): EMFSIVPMFEDVPPPAEVAMRNAPYWSRAVESQWMGLNFFPPQLEIECVS…


tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/2.61G [00:00<?, ?B/s]

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t33_650M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Embedding shape: (1280,)


# Chemberta

In [ ]:
# In your Jupyter cell:

import time
import random
import torch
from embpy.embedder import BioEmbedder

# 1) Initialize the embedder on CPU
embedder = BioEmbedder(device=torch.device("cpu"))
print("Available models:", embedder.list_available_models())

model_name = "chemberta_zinc_v1"
print(f"Using model: {model_name}")


# 2) Helper to generate a toy “SMILES” string of length n (just C/N/O here)
def random_smiles(n: int) -> str:  # noqa: D103
    tokens = ["C", "N", "O"]
    return "".join(random.choices(tokens, k=n))


# single‐sequence benchmark
smiles = random_smiles(128)
print("Random SMILES:", smiles)

start = time.time()
embedding = embedder.embed_molecule(smiles, model=model_name, pooling_strategy="mean")
dt = time.time() - start

print(f"Single embed → shape {embedding.shape}, took {dt:.3f}s")

# batch benchmark
batch = [random_smiles(128) for _ in range(16)]
start = time.time()
embeddings = embedder.embed_molecules_batch(batch, model=model_name, pooling_strategy="mean")
dt = time.time() - start

print(f"Batch of {len(batch)} → {[e.shape for e in embeddings]}\ntook {dt:.3f}s")

Available models: ['bert_base_uncased', 'borzoi_v0', 'borzoi_v1', 'chemberta_zinc_v1', 'enformer_human_rough', 'esm2_150M', 'esm2_35M', 'esm2_3B', 'esm2_650M', 'esm2_8M', 'minilm_l6_v2']
Using model: chemberta_zinc_v1
Random SMILES: CNOOCNCCCNONOCCCOCNNONOONOCOCNCCNCNCNCNCOONOCCCNOCONCCONNCCCCCCNCNONNOCCNCCCOOOOONCNCONCCOONNNNCNCNCCCNOONONOCNOOOCNOCNCCONCCOOC
Attention mask shape: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1]])
Single embed → shape (768,), took 0.700s
Attention mask shape: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1]])
Attention mask shape: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

# MolFormer

In [1]:
from embpy.embedder import BioEmbedder
import torch

embedder = BioEmbedder(device=torch.device("cpu"))
# should list both chemberta_zinc_v1 and molformer_xl
print(embedder.list_available_models())

smiles = "CC(=O)Oc1ccccc1C(=O)O"
emb = embedder.embed_molecule(smiles, model="molformer_base", pooling_strategy="cls")
print("MolFormer embedding shape:", emb.shape)

['bert_base_uncased', 'borzoi_v0', 'borzoi_v1', 'chemberta_zinc_v1', 'enformer_human_rough', 'esm2_150M', 'esm2_35M', 'esm2_3B', 'esm2_650M', 'esm2_8M', 'minilm_l6_v2', 'molformer_base']


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


MolFormer embedding shape: (768,)


In [4]:
print(emb)

[ 5.94280064e-01  4.52650040e-01  3.43747914e-01  6.67825580e-01
 -6.99579656e-01  2.02832788e-01 -5.44053353e-02  8.19928646e-01
  2.11379856e-01 -1.15139507e-01 -6.97321951e-01  7.15679109e-01
  5.20055711e-01  2.29307845e-01 -4.24501389e-01  1.97534814e-01
  3.03925332e-02 -2.96359032e-01 -4.01238978e-01 -5.29726088e-01
 -7.38125205e-01  6.60815015e-02  3.13309371e-01  7.46078014e-01
  2.41676614e-01  4.75266218e-01 -1.43960690e+00 -2.91483134e-01
 -6.32730305e-01  7.32680500e-01 -6.61421657e-01 -6.26281261e-01
 -1.05253361e-01  3.25632334e-01  6.07707977e-01  4.24847901e-01
  3.78585994e-01 -3.01597148e-01 -6.80134296e-01 -1.79451853e-01
 -1.41719967e-01  4.31221336e-01 -1.47480192e-02 -1.45637870e-01
 -1.72145277e-01 -2.95462102e-01  1.42974734e-01 -7.01401532e-02
  2.54375458e-01  4.04036850e-01  6.48585916e-01  1.57276704e-03
  5.96743226e-01  7.34008610e-01  1.47559628e-01 -1.99199796e-01
 -3.74050140e-01  2.49511153e-01 -1.34057164e-01 -5.61859123e-02
  3.31139356e-01  4.65937

In [3]:
import torch
from transformers import AutoModel, AutoTokenizer

model = AutoModel.from_pretrained("ibm/MoLFormer-XL-both-10pct", deterministic_eval=True, trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained("ibm/MoLFormer-XL-both-10pct", trust_remote_code=True)

smiles = "CC(=O)Oc1ccccc1C(=O)O"
inputs = tokenizer(smiles, padding=True, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)
outputs.pooler_output

tensor([[ 5.9428e-01,  4.5265e-01,  3.4375e-01,  6.6783e-01, -6.9958e-01,
          2.0283e-01, -5.4405e-02,  8.1993e-01,  2.1138e-01, -1.1514e-01,
         -6.9732e-01,  7.1568e-01,  5.2006e-01,  2.2931e-01, -4.2450e-01,
          1.9753e-01,  3.0393e-02, -2.9636e-01, -4.0124e-01, -5.2973e-01,
         -7.3813e-01,  6.6082e-02,  3.1331e-01,  7.4608e-01,  2.4168e-01,
          4.7527e-01, -1.4396e+00, -2.9148e-01, -6.3273e-01,  7.3268e-01,
         -6.6142e-01, -6.2628e-01, -1.0525e-01,  3.2563e-01,  6.0771e-01,
          4.2485e-01,  3.7859e-01, -3.0160e-01, -6.8013e-01, -1.7945e-01,
         -1.4172e-01,  4.3122e-01, -1.4748e-02, -1.4564e-01, -1.7215e-01,
         -2.9546e-01,  1.4297e-01, -7.0140e-02,  2.5438e-01,  4.0404e-01,
          6.4859e-01,  1.5728e-03,  5.9674e-01,  7.3401e-01,  1.4756e-01,
         -1.9920e-01, -3.7405e-01,  2.4951e-01, -1.3406e-01, -5.6186e-02,
          3.3114e-01,  4.6594e-02,  5.4272e-01, -9.9487e-02,  4.9468e-01,
         -2.7091e-02,  2.3985e-01,  1.